# IOAI — 2024 Selection Test Cv Cifar Resnet (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CIFAR-10 은 노트북이 torchvision 으로 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# CIFAR-10 이미지 분류 (ResNet) — 모범답안

Singapore IOAI 2024 — Selection Test (CV). CIFAR-10 을 **ResNet18**(ImageNet 사전학습)로 분류한다. 점수 = **테스트 정확도**. 노트북이
CIFAR-10 을 내려받고 학습한 뒤 **표준 순서(shuffle=False)** 로 테스트를 예측 → `submission.csv`(id,label) 제출.

**모범답안 = 사전학습 ResNet18 미세조정**: 스캐폴드는 학습 루프가 없어(미세조정 안 함) 정확도가 무작위(~10%)다.
아래처럼 **입력을 64px 로 업샘플**(사전학습 특징이 32px 엔 다소 안 맞음) + **SGD OneCycle·라벨스무딩·AMP** 로
10에폭 미세조정하면 테스트 정확도가 **≈0.93** 로 오른다. (32px 그대로면 ~0.87 에서 정체 — 업샘플이 핵심.)


In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)


In [ ]:
# 데이터 — CIFAR-10 (다운로드). 사전학습 특징을 살리려 64px 로 업샘플. 테스트는 표준 순서(shuffle=False).
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
RES = 64
tf_train = transforms.Compose([transforms.Resize(RES), transforms.RandomCrop(RES, padding=RES//8),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_test = transforms.Compose([transforms.Resize(RES), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_ds = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_test)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=4)
print("train", len(train_ds), "test", len(test_ds))


In [ ]:
# 모델 — ResNet18(ImageNet 사전학습), fc 를 10클래스로 교체
def build_model():
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)
model = build_model()

EPOCHS = 10
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.05, epochs=EPOCHS, steps_per_epoch=len(train_dl))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def test_accuracy():
    model.eval(); correct = total = 0
    for x, y in test_dl:
        with torch.cuda.amp.autocast(): p = model(x.to(device)).argmax(1).cpu()
        correct += (p == y).sum().item(); total += y.size(0)
    return correct / total


In [ ]:
# 학습 (SGD + OneCycle + 라벨스무딩 + AMP). T4 에서 ~수십 분.
for ep in range(EPOCHS):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(): loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    if (ep+1) % 2 == 0 or ep == EPOCHS-1:
        print(f"epoch {ep+1}/{EPOCHS}  test accuracy {test_accuracy():.4f}", flush=True)


In [ ]:
# 테스트 예측(표준 순서) -> submission.csv
model.eval(); preds = []
with torch.no_grad():
    for x, _ in test_dl:
        with torch.cuda.amp.autocast(): preds.append(model(x.to(device)).argmax(1).cpu().numpy())
preds = np.concatenate(preds)
pd.DataFrame({"id": range(len(preds)), "label": preds}).to_csv("submission.csv", index=False)
print("submission.csv 저장:", len(preds), "| 최종 테스트 정확도", round(test_accuracy(), 4))


### 정리
- 사전학습 ResNet18 + **64px 업샘플** + 10에폭 미세조정(SGD OneCycle·라벨스무딩·AMP) → 테스트 정확도 ≈ **0.93**.
- **핵심**: 32px 원본은 ImageNet 사전학습 특징과 해상도가 안 맞아 ~0.87 에서 정체 → 64~224px 업샘플이 큰 차이.
- **더 끌어올리려면**: 224px·에폭↑·Mixup/CutMix·TTA·앙상블·Cosine 재시작.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)